In [1]:
import pandas as pd
from sqlalchemy import create_engine
import os

# Create database file in the data folder
engine = create_engine("sqlite:///../data/olist.db")

# All 9 files and their table names
files = {
    "orders":      "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "products":    "olist_products_dataset.csv",
    "categories":  "product_category_name_translation.csv",
    "reviews":     "olist_order_reviews_dataset.csv",
    "customers":   "olist_customers_dataset.csv",
    "sellers":     "olist_sellers_dataset.csv",
    "payments":    "olist_order_payments_dataset.csv",
    "geo":         "olist_geolocation_dataset.csv"
}

# Load each file into the database
for table_name, filename in files.items():
    df = pd.read_csv(f"../data/{filename}")
    df.to_sql(table_name, engine, if_exists="replace", index=False)
    print(f"✓ {table_name}: {len(df):,} rows loaded")

print("\nAll done! Database created at data/olist.db")

✓ orders: 99,441 rows loaded
✓ order_items: 112,650 rows loaded
✓ products: 32,951 rows loaded
✓ categories: 71 rows loaded
✓ reviews: 99,224 rows loaded
✓ customers: 99,441 rows loaded
✓ sellers: 3,095 rows loaded
✓ payments: 103,886 rows loaded
✓ geo: 1,000,163 rows loaded

All done! Database created at data/olist.db


In [2]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("sqlite:///../data/olist.db")

query = """
SELECT
    o.order_id,
    o.order_purchase_timestamp,
    i.price,
    i.freight_value,
    t.product_category_name_english AS category,
    r.review_score
FROM orders o
JOIN order_items i ON o.order_id = i.order_id
JOIN products p ON i.product_id = p.product_id
JOIN categories t ON p.product_category_name = t.product_category_name
LEFT JOIN reviews r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
"""

df = pd.read_sql(query, engine)
print(f"Total rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print(f"\nDate range: {df['order_purchase_timestamp'].min()} to {df['order_purchase_timestamp'].max()}")
print(f"Price range: R${df['price'].min():.2f} to R${df['price'].max():.2f}")
print(f"Categories: {df['category'].nunique()} unique")
df.head()

Total rows: 109,273
Columns: ['order_id', 'order_purchase_timestamp', 'price', 'freight_value', 'category', 'review_score']

Date range: 2016-09-15 12:16:38 to 2018-08-29 15:00:37
Price range: R$0.85 to R$6735.00
Categories: 71 unique


,order_id,order_purchase_timestamp,price,freight_value,category,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,29.99,8.72,housewares,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,118.70,22.76,perfumery,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,159.90,19.22,auto,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,45.00,27.20,pet_shop,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,19.90,8.72,stationery,5.0


In [3]:
import numpy as np

# Clean the data
df = df[df['price'] > 0]  # remove zero prices
p99 = df['price'].quantile(0.99)
df = df[df['price'] < p99]  # remove top 1% outliers

# Parse date and create year_month column
df['order_date'] = pd.to_datetime(df['order_purchase_timestamp'])
df['year_month'] = df['order_date'].dt.to_period('M')

# Save as parquet file for fast loading later
df.to_parquet('../data/clean_data.parquet', index=False)

print(f"Clean dataset saved: {len(df):,} rows")
print(f"Removed {109273 - len(df):,} outlier rows")

Clean dataset saved: 108,175 rows
Removed 1,098 outlier rows
